In [1]:
# ============================================================
# EXP-1B) Phase necessity (Ours only)
#   - 6 activities (MHEALTH): act_id = [6,7,8,10,11,12]
#   - LOSO per activity
#   - test-time tempo warp mult sweep
#   - Quantify: phase usage stability (ΔeffK) ↔ count error (MAPE) correlation
#   - NO visualization
#   - Output: phase_rows.csv + corr_summary.txt (+ optional per-activity csv)
# ============================================================

import os, glob, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from scipy.stats import pearsonr, spearmanr


# ------------------------------------------------------------
# 1) Strict Seeding
# ------------------------------------------------------------
def set_strict_seed(seed: int):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ------------------------------------------------------------
# 2) Data Loading (MHEALTH)
# ------------------------------------------------------------
def load_mhealth_dataset(data_dir, target_activities_map, column_names):
    full_dataset = {}
    file_list = sorted(glob.glob(os.path.join(data_dir, "mHealth_subject*.log")))
    if not file_list:
        print(f"[Warning] No mHealth logs found in {data_dir}")
        return {}

    for file_path in file_list:
        file_name = os.path.basename(file_path)
        subj_part = file_name.split('.')[0]
        try:
            subj_id_num = int(''.join(filter(str.isdigit, subj_part)))
            subj_key = f"subject{subj_id_num}"
        except:
            subj_key = subj_part

        try:
            df = pd.read_csv(file_path, sep="\t", header=None)
            df = df.iloc[:, :len(column_names)]
            df.columns = column_names

            subj_data = {}
            for label_code, activity_name in target_activities_map.items():
                activity_df = df[df["activity_id"] == label_code].copy()
                if not activity_df.empty:
                    subj_data[activity_name] = activity_df.drop(columns=["activity_id"])
            full_dataset[subj_key] = subj_data
        except Exception as e:
            print(f"[Error] loading {file_name}: {e}")
            continue

    return full_dataset


def prepare_trial_list(label_config, full_data, target_map, feature_map):
    trial_list = []
    for subj, act_id, gt_count in label_config:
        act_name = target_map.get(act_id)
        feats = feature_map.get(act_id)

        if subj in full_data and act_name in full_data[subj]:
            raw_df = full_data[subj][act_name][feats]
            raw_np = raw_df.values.astype(np.float32)

            # trial-wise z-score
            mean = raw_np.mean(axis=0)
            std  = raw_np.std(axis=0) + 1e-6
            norm_np = (raw_np - mean) / std

            trial_list.append({
                "data": norm_np,         # (T,C)
                "count": float(gt_count),
                "subj": subj,
                "act_id": int(act_id),
                "meta": f"{subj}_{act_name}",
            })
    return trial_list


# ------------------------------------------------------------
# 3) Windowing (train labels: window count from trial-average rate)
# ------------------------------------------------------------
def trial_list_to_windows(trial_list, fs, win_sec=8.0, stride_sec=4.0, drop_last=True):
    win_len = int(round(win_sec * fs))
    stride  = int(round(stride_sec * fs))
    assert win_len > 0 and stride > 0

    windows = []
    for item in trial_list:
        x = item["data"]
        T = x.shape[0]
        total_count = float(item["count"])
        meta = item["meta"]

        total_dur = max(T / float(fs), 1e-6)
        rate_trial = total_count / total_dur  # reps/s

        if T < win_len:
            win_dur = T / float(fs)
            windows.append({"data": x, "count": rate_trial * win_dur, "meta": f"{meta}__win[0:{T}]"})
            continue

        starts = list(range(0, T - win_len + 1, stride))
        for st in starts:
            ed = st + win_len
            win_dur = win_len / float(fs)
            windows.append({"data": x[st:ed], "count": rate_trial * win_dur, "meta": f"{meta}__win[{st}:{ed}]"})
    return windows


# ------------------------------------------------------------
# 4) Test-time tempo warp
# ------------------------------------------------------------
def time_warp_linear(x_np: np.ndarray, dur_mult: float) -> np.ndarray:
    T, C = x_np.shape
    new_T = int(round(T * float(dur_mult)))
    new_T = max(new_T, 2)

    t_old = np.linspace(0.0, 1.0, T, dtype=np.float32)
    t_new = np.linspace(0.0, 1.0, new_T, dtype=np.float32)

    out = np.zeros((new_T, C), dtype=np.float32)
    for c in range(C):
        out[:, c] = np.interp(t_new, t_old, x_np[:, c]).astype(np.float32)
    return out


# ------------------------------------------------------------
# 5) Dataset / Collate
# ------------------------------------------------------------
class TrialDataset(Dataset):
    def __init__(self, windows_list):
        self.items = windows_list

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        x = torch.tensor(item["data"], dtype=torch.float32).transpose(0, 1)  # (C,T)
        y = torch.tensor(item["count"], dtype=torch.float32)                 # window count
        return x, y


def collate_variable_length(batch):
    max_len = max([x[0].shape[1] for x in batch])
    C = batch[0][0].shape[0]

    xs, masks, ys, lengths = [], [], [], []
    for x, y in batch:
        T = x.shape[1]
        lengths.append(T)
        pad = max_len - T
        if pad > 0:
            x = torch.cat([x, torch.zeros(C, pad)], dim=1)
            m = torch.cat([torch.ones(T), torch.zeros(pad)], dim=0)
        else:
            m = torch.ones(T)
        xs.append(x); masks.append(m); ys.append(y)

    return {
        "data": torch.stack(xs),                 # (B,C,Tmax)
        "mask": torch.stack(masks),              # (B,Tmax)
        "count": torch.stack(ys),                # (B,)
        "length": torch.tensor(lengths, dtype=torch.float32)  # (B,)
    }


# ------------------------------------------------------------
# 6) Model (OURS)
# ------------------------------------------------------------
class ManifoldEncoder(nn.Module):
    def __init__(self, input_ch, hidden_dim=128, latent_dim=16, n_blocks=2):
        super().__init__()
        layers, in_ch = [], input_ch
        for _ in range(n_blocks):
            layers += [nn.Conv1d(in_ch, hidden_dim, 5, padding=2), nn.ReLU()]
            in_ch = hidden_dim
        layers += [nn.Conv1d(hidden_dim, latent_dim, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        z = self.net(x)           # (B,D,T)
        return z.transpose(1, 2)  # (B,T,D)


class ManifoldDecoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, out_ch, n_blocks=2):
        super().__init__()
        layers, in_ch = [], latent_dim
        for _ in range(n_blocks):
            layers += [nn.Conv1d(in_ch, hidden_dim, 5, padding=2), nn.ReLU()]
            in_ch = hidden_dim
        layers += [nn.Conv1d(hidden_dim, out_ch, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        return self.net(z.transpose(1, 2))  # (B,C,T)


class MultiRateHead(nn.Module):
    def __init__(self, latent_dim=16, hidden=128, K_max=6):
        super().__init__()
        self.K_max = K_max
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1 + K_max)  # [amp | phase_logits...]
        )

    def forward(self, z, tau=1.0):
        out = self.net(z)                   # (B,T,1+K)
        amp = F.softplus(out[..., 0])       # (B,T)
        phase_logits = out[..., 1:]         # (B,T,K)
        phase = F.softmax(phase_logits / tau, dim=-1)
        return amp, phase, phase_logits


class KAutoCountModel(nn.Module):
    def __init__(self, input_ch, hidden_dim=128, latent_dim=16, K_max=6, enc_blocks=2, dec_blocks=2):
        super().__init__()
        self.encoder = ManifoldEncoder(input_ch, hidden_dim, latent_dim, n_blocks=enc_blocks)
        self.decoder = ManifoldDecoder(latent_dim, hidden_dim, input_ch, n_blocks=dec_blocks)
        self.rate_head = MultiRateHead(latent_dim, hidden=hidden_dim, K_max=K_max)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
        with torch.no_grad():
            b = self.rate_head.net[-1].bias
            b.zero_()
            b[0].fill_(-2.0)

    def forward(self, x, mask=None, tau=1.0):
        z = self.encoder(x)              # (B,T,D)
        _ = self.decoder(z)              # not used in this exp, but kept

        amp_t, phase_p, _ = self.rate_head(z, tau=tau)  # amp_t: (B,T), phase_p: (B,T,K)
        micro_rate_t = amp_t

        # p_bar (B,K)
        p_bar = phase_p.mean(dim=1)

        # effK (B,)
        effK = 1.0 / (p_bar.pow(2).sum(dim=1) + 1e-6)

        # rep_rate_t (B,T)
        rep_rate_t = micro_rate_t / (effK.unsqueeze(1) + 1e-6)

        avg_rep_rate = rep_rate_t.mean(dim=1)  # (B,)

        aux = {"phase_p": phase_p, "effK": effK}
        return avg_rep_rate, aux


# ------------------------------------------------------------
# 7) Train (OURS) - same as EXP-1 supervision (rate from window count)
# ------------------------------------------------------------
def train_one_epoch_ours(model, loader, optimizer, config, device):
    model.train()
    fs = config["fs"]
    tau = config.get("tau", 1.0)

    for batch in loader:
        x = batch["data"].to(device)
        y_count = batch["count"].to(device)
        length = batch["length"].to(device)

        duration = torch.clamp(length / fs, min=1e-6)
        y_rate = y_count / duration

        optimizer.zero_grad()
        rate_hat, _ = model(x, mask=None, tau=tau)

        loss = F.mse_loss(rate_hat, y_rate)
        loss.backward()
        optimizer.step()


def mae_mape(pred, gt, eps=1e-6):
    mae = abs(float(pred) - float(gt))
    mape = mae / (abs(float(gt)) + eps) * 100.0
    return mae, mape


# ------------------------------------------------------------
# 8) Run
# ------------------------------------------------------------
def main():
    CONFIG = {
        "seed": 42,
        "data_dir": "/content/drive/MyDrive/Colab Notebooks/HAR_data/MHEALTHDATASET",
        "fs": 50,
        "epochs": 100,
        "lr": 5e-4,
        "batch_size": 64,
        "win_sec": 8.0,
        "stride_sec": 4.0,
        "hidden_dim": 128,
        "latent_dim": 16,
        "K_max": 6,
        "enc_blocks": 2,
        "dec_blocks": 2,
        "tau": 1.0,
        "COLUMN_NAMES": [
            'acc_chest_x', 'acc_chest_y', 'acc_chest_z',
            'ecg_1', 'ecg_2',
            'acc_ankle_x', 'acc_ankle_y', 'acc_ankle_z',
            'gyro_ankle_x', 'gyro_ankle_y', 'gyro_ankle_z',
            'mag_ankle_x', 'mag_ankle_y', 'mag_ankle_z',
            'acc_arm_x', 'acc_arm_y', 'acc_arm_z',
            'gyro_arm_x', 'gyro_arm_y', 'gyro_arm_z',
            'mag_arm_x', 'mag_arm_y', 'mag_arm_z',
            'activity_id'
        ]
    }

    MULTS = [0.6, 0.7, 0.75, 0.8, 0.9, 1.0, 1.1, 1.25, 1.4, 1.6]

    ACTIVITY_SPECS = [
        {"act_id": 6,  "act_name": "Waist bends forward",       "labels": [
            ("subject1", 6, 21), ("subject2", 6, 19), ("subject3", 6, 21), ("subject4", 6, 20), ("subject5", 6, 20),
            ("subject6", 6, 20), ("subject7", 6, 20), ("subject8", 6, 21), ("subject9", 6, 21), ("subject10", 6, 20),
        ]},
        {"act_id": 7,  "act_name": "Frontal elevation of arms", "labels": [
            ("subject1", 7, 20), ("subject2", 7, 20), ("subject3", 7, 20), ("subject4", 7, 20), ("subject5", 7, 20),
            ("subject6", 7, 20), ("subject7", 7, 20), ("subject8", 7, 19), ("subject9", 7, 19), ("subject10", 7, 20),
        ]},
        {"act_id": 8,  "act_name": "Knees bending",             "labels": [
            ("subject1", 8, 20), ("subject2", 8, 21), ("subject3", 8, 21), ("subject4", 8, 19), ("subject5", 8, 20),
            ("subject6", 8, 20), ("subject7", 8, 21), ("subject8", 8, 21), ("subject9", 8, 21), ("subject10", 8, 21),
        ]},
        {"act_id": 12, "act_name": "Jump front & back",         "labels": [
            ("subject1", 12, 20), ("subject2", 12, 22), ("subject3", 12, 21), ("subject4", 12, 21), ("subject5", 12, 20),
            ("subject6", 12, 21), ("subject7", 12, 19), ("subject8", 12, 20), ("subject9", 12, 20), ("subject10", 12, 20),
        ]},
        {"act_id": 10, "act_name": "Jogging",                   "labels": [
            ("subject1", 10, 157), ("subject2", 10, 161), ("subject3", 10, 154), ("subject4", 10, 154), ("subject5", 10, 160),
            ("subject6", 10, 156), ("subject7", 10, 153), ("subject8", 10, 160), ("subject9", 10, 166), ("subject10", 10, 156),
        ]},
        {"act_id": 11, "act_name": "Running",                   "labels": [
            ("subject1", 11, 165), ("subject2", 11, 158), ("subject3", 11, 174), ("subject4", 11, 163), ("subject5", 11, 157),
            ("subject6", 11, 172), ("subject7", 11, 149), ("subject8", 11, 166), ("subject9", 11, 174), ("subject10", 11, 172),
        ]},
    ]

    COMMON_FEATURES = [
        'acc_chest_x', 'acc_chest_y', 'acc_chest_z',
        'acc_ankle_x', 'acc_ankle_y', 'acc_ankle_z',
        'gyro_ankle_x', 'gyro_ankle_y', 'gyro_ankle_z',
        'acc_arm_x', 'acc_arm_y', 'acc_arm_z',
        'gyro_arm_x', 'gyro_arm_y', 'gyro_arm_z'
    ]

    TARGET_ACTIVITIES_MAP = {spec["act_id"]: spec["act_name"] for spec in ACTIVITY_SPECS}
    ACT_FEATURE_MAP = {spec["act_id"]: COMMON_FEATURES for spec in ACTIVITY_SPECS}

    subjects = [f"subject{i}" for i in range(1, 11)]
    OUT_DIR = "exp1b_phase_outputs"
    os.makedirs(OUT_DIR, exist_ok=True)

    set_strict_seed(CONFIG["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Device] {device}")

    full_data = load_mhealth_dataset(CONFIG["data_dir"], TARGET_ACTIVITIES_MAP, CONFIG["COLUMN_NAMES"])
    if not full_data:
        print("[Abort] dataset not loaded.")
        return

    rows = []

    for spec in ACTIVITY_SPECS:
        act_id = int(spec["act_id"])
        act_name = spec["act_name"]
        labels_all = spec["labels"]

        for test_subj in subjects:
            set_strict_seed(CONFIG["seed"])

            train_subjects = [s for s in subjects if s != test_subj]
            train_labels = [(subj, act_id, gt) for (subj, _, gt) in labels_all if subj in train_subjects]
            test_labels  = [(subj, act_id, gt) for (subj, _, gt) in labels_all if subj == test_subj]

            train_trials = prepare_trial_list(train_labels, full_data, TARGET_ACTIVITIES_MAP, ACT_FEATURE_MAP)
            test_trials  = prepare_trial_list(test_labels,  full_data, TARGET_ACTIVITIES_MAP, ACT_FEATURE_MAP)
            if len(train_trials) == 0 or len(test_trials) == 0:
                continue

            train_windows = trial_list_to_windows(
                train_trials, fs=CONFIG["fs"],
                win_sec=CONFIG["win_sec"], stride_sec=CONFIG["stride_sec"]
            )
            train_loader = DataLoader(
                TrialDataset(train_windows),
                batch_size=CONFIG["batch_size"],
                shuffle=True,
                collate_fn=collate_variable_length,
                num_workers=0
            )

            input_ch = train_windows[0]["data"].shape[1]
            model = KAutoCountModel(
                input_ch=input_ch,
                hidden_dim=CONFIG["hidden_dim"],
                latent_dim=CONFIG["latent_dim"],
                K_max=CONFIG["K_max"],
                enc_blocks=CONFIG["enc_blocks"],
                dec_blocks=CONFIG["dec_blocks"]
            ).to(device)

            opt = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])
            for _ in range(CONFIG["epochs"]):
                train_one_epoch_ours(model, train_loader, opt, CONFIG, device)

            model.eval()

            # test trial
            x_raw = test_trials[0]["data"]
            gt_count = float(test_trials[0]["count"])

            effK_ref = None

            for mult in MULTS:
                x_warp = time_warp_linear(x_raw, float(mult))
                T = x_warp.shape[0]
                dur = T / float(CONFIG["fs"])

                x_tensor = torch.tensor(x_warp, dtype=torch.float32).transpose(0,1).unsqueeze(0).to(device)

                with torch.no_grad():
                    rate_hat, aux = model(x_tensor, mask=None, tau=CONFIG["tau"])
                    effK = float(aux["effK"].item())

                pred_count = float(rate_hat.item() * dur)
                _, mape = mae_mape(pred_count, gt_count)

                if abs(mult - 1.0) < 1e-9:
                    effK_ref = effK

                if effK_ref is not None:
                    rows.append({
                        "act_id": act_id,
                        "act_name": act_name,
                        "subject": test_subj,
                        "mult": float(mult),
                        "effK": effK,
                        "delta_effK": abs(effK - effK_ref),
                        "MAPE": float(mape)
                    })

            print(f"[Done] {act_name} | test={test_subj}")

    df = pd.DataFrame(rows)
    out_csv = os.path.join(OUT_DIR, "phase_rows.csv")
    df.to_csv(out_csv, index=False)
    print(f"\n[Saved] rows -> {out_csv}")

    # Overall correlation
    pear_r, pear_p = pearsonr(df["delta_effK"], df["MAPE"])
    sp_r, sp_p     = spearmanr(df["delta_effK"], df["MAPE"])

    # Per-activity correlation (optional but useful)
    per_act = []
    for (aid, aname), g in df.groupby(["act_id", "act_name"]):
        if len(g) < 5:
            continue
        r1, p1 = pearsonr(g["delta_effK"], g["MAPE"])
        r2, p2 = spearmanr(g["delta_effK"], g["MAPE"])
        per_act.append({
            "act_id": aid, "act_name": aname,
            "pearson_r": r1, "pearson_p": p1,
            "spearman_r": r2, "spearman_p": p2,
            "n": len(g)
        })
    df_per = pd.DataFrame(per_act).sort_values(["act_id"]).reset_index(drop=True)
    per_csv = os.path.join(OUT_DIR, "corr_per_activity.csv")
    df_per.to_csv(per_csv, index=False)

    summary_txt = os.path.join(OUT_DIR, "corr_summary.txt")
    with open(summary_txt, "w") as f:
        f.write("EXP-1B Phase stability vs Count error (OURS only)\n")
        f.write(f"Overall Pearson r={pear_r:.4f}, p={pear_p:.3e}\n")
        f.write(f"Overall Spearman r={sp_r:.4f}, p={sp_p:.3e}\n")
        f.write("\nPer-activity correlations saved: corr_per_activity.csv\n")

    print("\n[Overall correlation]")
    print(f"Pearson  r = {pear_r:.4f} (p={pear_p:.3e})")
    print(f"Spearman ρ = {sp_r:.4f} (p={sp_p:.3e})")
    print(f"[Saved] summary -> {summary_txt}")
    print(f"[Saved] per-activity -> {per_csv}")


if __name__ == "__main__":
    main()


[Device] cuda
[Done] Waist bends forward | test=subject1
[Done] Waist bends forward | test=subject2
[Done] Waist bends forward | test=subject3
[Done] Waist bends forward | test=subject4
[Done] Waist bends forward | test=subject5
[Done] Waist bends forward | test=subject6
[Done] Waist bends forward | test=subject7
[Done] Waist bends forward | test=subject8
[Done] Waist bends forward | test=subject9
[Done] Waist bends forward | test=subject10
[Done] Frontal elevation of arms | test=subject1
[Done] Frontal elevation of arms | test=subject2
[Done] Frontal elevation of arms | test=subject3
[Done] Frontal elevation of arms | test=subject4
[Done] Frontal elevation of arms | test=subject5
[Done] Frontal elevation of arms | test=subject6
[Done] Frontal elevation of arms | test=subject7
[Done] Frontal elevation of arms | test=subject8
[Done] Frontal elevation of arms | test=subject9
[Done] Frontal elevation of arms | test=subject10
[Done] Knees bending | test=subject1
[Done] Knees bending | test